# PrimeVul Score Conversion

Converts lm-eval generate_until output into router training data.
Same structure as convert_dataset_7_model.ipynb.

Paper Eq. 1: s_i^(t) = (1/M) * sum_m evaluate(y_hat, y)  — with repeats=5, score = fraction correct.

In [ ]:
import json
import os
import glob
import random
import pandas as pd
from collections import defaultdict

random.seed(42)

output_file_path = "./datasets/split2_primevul_gen"
os.makedirs(output_file_path, exist_ok=True)

# [org, model_name, output_dir_name]
# org/model_name  →  HuggingFace model ID used as key in scores dict
# output_dir_name →  directory name under output/primevul_gen_1to2/
model_list = [
    ["codellama",       "CodeLlama-7b-Instruct-hf",           "CodeLlama-7b-Instruct"],
    ["codellama",       "CodeLlama-13b-Instruct-hf",          "CodeLlama-13b-Instruct"],
    ["deepseek-ai",     "DeepSeek-Coder-V2-Lite-Instruct",    "DeepSeek-Coder-V2-Lite-Instruct"],
    ["Qwen",            "Qwen2.5-Coder-14B-Instruct",         "Qwen2.5-Coder-14B-Instruct"],
    ["bigcode",         "starcoder2-15b-instruct-v0.1",       "starcoder2-15b-instruct"],
    ["Virtue-AI-HUB",  "VulnLLM-R-7B",                       "VulnLLM-R-7B"],
]

# PrimeVul (generate_until)

In [50]:
import re

def extract_answer(raw: str) -> str | None:
    """Mirror lm-eval 'extract-answer': pull Yes/No from \\boxed{...}."""
    m = re.search(r"\\boxed\{(Yes|No)", raw, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    m = re.search(r"\b(Yes|No)\b", raw, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    return None   # malformed / unrecognised


# scores_by_idx[idx][model_id] = acc
# funcs_by_idx[idx]            = func text
scores_by_idx = {}
funcs_by_idx  = {}

for model_index, model_info in enumerate(model_list):
    model_pre  = model_info[0]
    model      = model_info[1]
    dir_name   = model_info[2]
    model_id   = model_pre + "/" + model

    pattern = f"./output/primevul_gen_1to2/{dir_name}/*/samples_primevul_gen_1to2_*.jsonl"
    files = glob.glob(pattern)
    if not files:
        print(f"[skip] {model}: no samples file at {pattern}")
        continue

    n_docs = 0
    avg_acc = 0.0

    with open(files[0]) as f:
        for line in f:
            if not line.strip():
                continue
            entry  = json.loads(line)

            # Use doc["idx"] — the true PrimeVul dataset index — as the stable
            # key so scores align correctly even if doc_id differs across runs.
            idx    = entry["doc"]["idx"]
            resps  = entry.get("resps", [[]])[0]   # all M=5 raw responses
            target = entry["target"].strip()        # "Yes" or "No"

            # Paper Eq. 1: s = (1/M) * sum evaluate(y_hat_m, y)
            # Divide by len(resps)=M so malformed counts as wrong and
            # scores stay exact multiples of 0.2
            answers = [extract_answer(r) for r in resps]
            valid   = [a for a in answers if a is not None]
            correct = sum(1 for a in valid if a == target)
            acc     = correct / len(resps) if resps else 0.0

            # store score keyed by stable idx
            if idx not in scores_by_idx:
                scores_by_idx[idx] = {}
            scores_by_idx[idx][model_id] = acc

            # store func text (same across models, only needed once)
            if idx not in funcs_by_idx:
                funcs_by_idx[idx] = entry["doc"]["func"]

            n_docs  += 1
            avg_acc += acc

    avg_acc /= n_docs if n_docs else 1
    print(f"[ok] {model:<45} docs={n_docs}  avg_score={avg_acc:.3f}")

# Build output_data sorted by idx for reproducibility
output_data = [
    {"question": funcs_by_idx[idx], "scores": scores_by_idx[idx]}
    for idx in sorted(scores_by_idx.keys())
]

print(f"\nTotal docs: {len(output_data)}")

[ok] CodeLlama-7b-Instruct-hf                      docs=18012  avg_score=0.395
[ok] CodeLlama-13b-Instruct-hf                     docs=18012  avg_score=0.661
[ok] DeepSeek-Coder-V2-Lite-Instruct               docs=18012  avg_score=0.678
[ok] Qwen2.5-Coder-14B-Instruct                    docs=18012  avg_score=0.712
[ok] starcoder2-15b-instruct-v0.1                  docs=18012  avg_score=0.483
[ok] VulnLLM-R-7B                                  docs=18012  avg_score=0.746

Total docs: 18012


In [ ]:
# Train / test split  70 / 30  (same as paper)
train_split_index = random.sample(range(len(output_data)), len(output_data))
output_data = [output_data[idx] for idx in train_split_index]

train_split = output_data[:int(0.7 * len(output_data))]
test_split  = output_data[int(0.7 * len(output_data)):]

with open(os.path.join(output_file_path, "primevul_gen_train.json"), "w") as f:
    json.dump(train_split, f)

with open(os.path.join(output_file_path, "primevul_gen_test.json"), "w") as f:
    json.dump(test_split, f)

print(f"train: {len(train_split)}  test: {len(test_split)}")
print(f"Saved to {output_file_path}/")

train: 12608  test: 5404
Saved to ./datasets/split2_primevul/


# Get ACC (per-model accuracy on full dataset)

In [52]:
with open(os.path.join(output_file_path, "primevul_train.json")) as f:
    output_data = json.load(f)

# Build score dict from model list
correct_dict = {m[0]+"/"+m[1]: 0 for m in model_list}

data_size = len(output_data)
for item in output_data:
    for key, score in item["scores"].items():
        if key in correct_dict:
            correct_dict[key] += score / data_size * 100

df = pd.DataFrame.from_dict(correct_dict, orient="index", columns=["accuracy (%)"])

# Also show how many queries have partial scores (0 < s < 1) per model
partial_dict = {m[0]+"/"+m[1]: 0 for m in model_list}
for item in output_data:
    for key, score in item["scores"].items():
        if key in partial_dict and 1e-9 < score < 1 - 1e-9:
            partial_dict[key] += 1

df["partial_scores"] = pd.Series(partial_dict)
df["partial_%"] = (df["partial_scores"] / data_size * 100).round(1)
df

,accuracy (%),partial_scores,partial_%
codellama/CodeLlama-7b-Instruct-hf,39.438452,5126,40.7
codellama/CodeLlama-13b-Instruct-hf,65.880393,2886,22.9
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct,67.625317,1420,11.3
Qwen/Qwen2.5-Coder-14B-Instruct,70.991434,3441,27.3
bigcode/starcoder2-15b-instruct-v0.1,48.272525,4965,39.4
Virtue-AI-HUB/VulnLLM-R-7B,74.270305,1098,8.7


In [53]:
with open(os.path.join(output_file_path, "primevul_train.json")) as f:
    output_data = json.load(f)

M = 5   # number of runs per query

model_ids = list(output_data[0]["scores"].keys())
total_queries = len(output_data)

print(f"{'Model':<55} {'correct':>10} {'total':>10} {'accuracy':>10}")
print("─" * 90)

for model_id in model_ids:
    # score = correct / M  →  correct = score * M
    total_correct = sum(item["scores"][model_id] * M for item in output_data)
    total_possible = total_queries * M
    accuracy = total_correct / total_possible * 100
    print(f"{model_id:<55} {total_correct:>10.0f} {total_possible:>10,} {accuracy:>9.2f}%")

Model                                                      correct      total   accuracy
──────────────────────────────────────────────────────────────────────────────────────────
codellama/CodeLlama-7b-Instruct-hf                           24862     63,040     39.44%
codellama/CodeLlama-13b-Instruct-hf                          41531     63,040     65.88%
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct                  42631     63,040     67.63%
Qwen/Qwen2.5-Coder-14B-Instruct                              44753     63,040     70.99%
bigcode/starcoder2-15b-instruct-v0.1                         30431     63,040     48.27%
Virtue-AI-HUB/VulnLLM-R-7B                                   46820     63,040     74.27%
